# Decision Tree Analysis for Heart Disease Prediction
**Machine Learning Assignment — Member 6**  
**Algorithm: Decision Tree Classifier (CART)**

---

## 1. Problem Definition

Predict whether a patient has heart disease using the Cleveland dataset. The model classifies patients into:
- **0** = no heart disease
- **1** = heart disease present

A **Decision Tree** recursively partitions the feature space by selecting the feature and threshold that best separates the classes at each node, using the **Gini impurity** criterion (CART algorithm). The result is a human-readable tree structure ideal for clinical interpretation.

## 2. Dataset Description

| Field | Value |
|---|---|
| Source | `Data_set/processed.cleveland.data` |
| Features | 13 clinical attributes |
| Samples | 303 instances |

### Features
| # | Feature | Description |
|---|---|---|
| 1 | `age` | Age in years |
| 2 | `sex` | Sex (1=male, 0=female) |
| 3 | `cp` | Chest pain type (1-4) |
| 4 | `trestbps` | Resting blood pressure |
| 5 | `chol` | Serum cholesterol |
| 6 | `fbs` | Fasting blood sugar > 120 mg/dl |
| 7 | `restecg` | Resting ECG (0-2) |
| 8 | `thalach` | Max heart rate achieved |
| 9 | `exang` | Exercise-induced angina |
| 10 | `oldpeak` | ST depression by exercise |
| 11 | `slope` | Slope of peak exercise ST |
| 12 | `ca` | Major vessels by fluoroscopy |
| 13 | `thal` | Thalassemia type |

## 3. Imports & Setup

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

ROOT_DIR = os.path.abspath(os.path.join('..'))
sys.path.append(ROOT_DIR)

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)

from src.data_preprocessing import load_data, preprocess_data, split_features_target
from src.evaluation import save_metrics

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('All imports successful')

## 4. Data Loading & Preprocessing

In [ ]:
DATA_PATH = os.path.join('..', 'Data_set', 'processed.cleveland.data')
raw_df    = load_data(DATA_PATH)
clean_df  = preprocess_data(raw_df)

print(f'Raw shape   : {raw_df.shape}')
print(f'Clean shape : {clean_df.shape}')
print(f'Missing values after preprocessing: {clean_df.isnull().sum().sum()}')
print(f'\nTarget distribution:\n{clean_df["target"].value_counts()}')
clean_df.head()

### 4.1 Exploratory Data Analysis

In [ ]:
RESULTS_DIR = os.path.join('..', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Box plots: feature distributions by target class
features = [c for c in clean_df.columns if c != 'target']
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(features):
    groups = [clean_df[clean_df['target']==0][feat].values,
              clean_df[clean_df['target']==1][feat].values]
    bp = axes[i].boxplot(groups, patch_artist=True,
                         boxprops=dict(facecolor='#90CAF9'),
                         medianprops=dict(color='#1565C0', linewidth=2))
    bp['boxes'][1].set(facecolor='#FFCDD2')
    axes[i].set_xticklabels(['No Disease', 'Disease'], fontsize=8)
    axes[i].set_title(feat, fontsize=10, fontweight='bold')

for j in range(len(features), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Feature Box Plots by Heart Disease Status', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'dt_boxplots.png'), bbox_inches='tight', dpi=100)
plt.show()

## 5. Why Decision Trees?

| Advantage | Explanation |
|---|---|
| Highly interpretable | Tree rules can be read by clinical staff |
| No feature scaling needed | Decision boundaries are axis-aligned thresholds |
| Handles non-linearity | Captures non-linear relationships naturally |
| Feature importance | Directly provides feature importance scores |
| Fast inference | O(log n) prediction time |

**CART algorithm** splits nodes by minimising **Gini impurity**:
$$Gini(t) = 1 - \sum_{c} p(c|t)^2$$

The best split is chosen to minimise the weighted Gini impurity of child nodes.

## 6. Train / Test Split

> Note: Decision Trees do **not** require feature scaling. StandardScaler is not applied here.

In [ ]:
X, y = split_features_target(clean_df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f'Train : {X_train.shape[0]} samples | Test : {X_test.shape[0]} samples')
print('Note: No feature scaling needed for Decision Trees')

## 7. Max Depth Tuning

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
depth_range = range(1, 16)
train_acc_list, cv_acc_list = [], []

for d in depth_range:
    dt_tmp = DecisionTreeClassifier(max_depth=d, criterion='gini', random_state=RANDOM_STATE)
    dt_tmp.fit(X_train, y_train)
    train_acc_list.append(dt_tmp.score(X_train, y_train))
    cv_scores = cross_val_score(dt_tmp, X, y, cv=cv, scoring='accuracy')
    cv_acc_list.append(cv_scores.mean())

best_depth = list(depth_range)[int(np.argmax(cv_acc_list))]
print(f'Best max_depth = {best_depth}  →  CV Accuracy = {max(cv_acc_list):.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(list(depth_range), train_acc_list, 'o-', color='#43A047',
        linewidth=2, label='Training Accuracy')
ax.plot(list(depth_range), cv_acc_list, 's-', color='#1976D2',
        linewidth=2, label='CV Accuracy')
ax.axvline(best_depth, color='#F44336', linestyle='--', linewidth=2,
           label=f'Best depth = {best_depth}')
ax.set_title('Max Depth vs Accuracy (Training vs CV)', fontsize=13, fontweight='bold')
ax.set_xlabel('max_depth', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_xticks(list(depth_range))
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'dt_depth_tuning.png'), dpi=120)
plt.show()

## 8. Model Training

In [ ]:
dt = DecisionTreeClassifier(
    max_depth=best_depth,
    criterion='gini',
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=RANDOM_STATE
)
dt.fit(X_train, y_train)

print('Decision Tree trained successfully')
print(f'  max_depth        : {dt.max_depth}')
print(f'  criterion        : {dt.criterion}')
print(f'  min_samples_split: {dt.min_samples_split}')
print(f'  min_samples_leaf : {dt.min_samples_leaf}')
print(f'  n_leaves         : {dt.get_n_leaves()}')
print(f'  actual depth     : {dt.get_depth()}')

## 9. Decision Tree Visualisation

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(
    dt,
    feature_names=X.columns.tolist(),
    class_names=['No Disease', 'Disease'],
    filled=True,
    rounded=True,
    fontsize=10,
    ax=ax,
    impurity=True,
    precision=3
)
ax.set_title(f'Decision Tree (max_depth={best_depth}) — Heart Disease Prediction',
             fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'dt_tree_plot.png'), bbox_inches='tight', dpi=120)
plt.show()
print('Tree visualisation saved')

In [ ]:
# Text representation of the tree rules
print('Decision Tree Rules:')
print('=' * 60)
tree_rules = export_text(dt, feature_names=X.columns.tolist())
print(tree_rules)

## 10. Model Evaluation

In [ ]:
y_pred       = dt.predict(X_test)
y_pred_proba = dt.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_pred_proba)
cm   = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print('=' * 55)
print('      DECISION TREE — EVALUATION METRICS')
print('=' * 55)
print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print(f'  ROC AUC   : {auc:.4f}')
print(f'  Sensitivity: {tp/(tp+fn):.4f}')
print(f'  Specificity: {tn/(tn+fp):.4f}')
print('=' * 55)

In [ ]:
print('CLASSIFICATION REPORT:')
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))
print('CONFUSION MATRIX:')
print(f'  TN={tn}  FP={fp}')
print(f'  FN={fn}  TP={tp}')

## 11. Results Visualisation

In [ ]:
fig = plt.figure(figsize=(14, 5))
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)

# Confusion Matrix
ax1 = fig.add_subplot(gs[0])
sns.heatmap(
    np.array([[tn, fp], [fn, tp]]),
    annot=True, fmt='d', cmap='Greens',
    xticklabels=['Predicted\nNo Disease', 'Predicted\nDisease'],
    yticklabels=['Actual\nNo Disease', 'Actual\nDisease'],
    linewidths=1, linecolor='white', ax=ax1,
    annot_kws={'size': 14, 'weight': 'bold'}
)
ax1.set_title(f'Confusion Matrix (depth={best_depth})', fontsize=13, fontweight='bold')

# ROC Curve
ax2 = fig.add_subplot(gs[1])
fpr_c, tpr_c, _ = roc_curve(y_test, y_pred_proba)
ax2.plot(fpr_c, tpr_c, color='#388E3C', linewidth=2.5,
         label=f'Decision Tree (AUC={auc:.4f})')
ax2.plot([0, 1], [0, 1], 'k--', linewidth=1)
ax2.fill_between(fpr_c, tpr_c, alpha=0.1, color='#388E3C')
ax2.set_title('ROC Curve', fontsize=13, fontweight='bold')
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.spines[['top', 'right']].set_visible(False)

plt.suptitle('Decision Tree — Evaluation Results', fontsize=14, fontweight='bold', y=1.02)
plt.savefig(os.path.join(RESULTS_DIR, 'dt_evaluation.png'), bbox_inches='tight', dpi=120)
plt.show()
print('Evaluation plots saved')

## 12. Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'Feature':   X.columns.tolist(),
    'Importance': dt.feature_importances_
}).sort_values('Importance', ascending=False)

print('Feature Importances (Gini-based):')
print('=' * 40)
print(importance_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colors  = ['#1B5E20' if i == 0 else '#66BB6A' for i in range(len(importance_df))]
bars    = ax.barh(importance_df['Feature'][::-1],
                  importance_df['Importance'][::-1],
                  color=colors[::-1], edgecolor='white', height=0.6)

for bar, val in zip(bars, importance_df['Importance'][::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

ax.set_title('Feature Importance — Decision Tree (Gini)', fontsize=12, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'dt_feature_importance.png'), dpi=120)
plt.show()

## 13. Cross-Validation

In [ ]:
cv_scores = cross_val_score(dt, X, y, cv=cv, scoring='accuracy')
cv_f1     = cross_val_score(dt, X, y, cv=cv, scoring='f1')
cv_auc    = cross_val_score(dt, X, y, cv=cv, scoring='roc_auc')

print('5-Fold Cross-Validation:')
print(f'  Accuracy : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'  F1-Score : {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')
print(f'  ROC AUC  : {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
folds = [f'Fold {i+1}' for i in range(len(cv_scores))]
bars  = ax.bar(folds, cv_scores, color='#388E3C', width=0.5, edgecolor='white')
ax.axhline(cv_scores.mean(), color='#F44336', linewidth=2, linestyle='--',
           label=f'Mean = {cv_scores.mean():.4f}')
for bar, val in zip(bars, cv_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylim(0.5, 1.05)
ax.set_title(f'5-Fold CV Accuracy — Decision Tree (depth={best_depth})',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'dt_cross_validation.png'), dpi=120)
plt.show()

## 14. Criterion Comparison (Gini vs Entropy)

In [ ]:
criterion_results = []
for crit in ['gini', 'entropy']:
    dt_tmp = DecisionTreeClassifier(max_depth=best_depth, criterion=crit,
                                    min_samples_split=5, min_samples_leaf=2,
                                    random_state=RANDOM_STATE)
    scores = cross_val_score(dt_tmp, X, y, cv=cv, scoring='accuracy')
    criterion_results.append({'Criterion': crit.capitalize(),
                               'Mean CV Acc': scores.mean(),
                               'Std': scores.std()})

crit_df = pd.DataFrame(criterion_results)
print('Criterion Comparison:')
print(crit_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(crit_df['Criterion'], crit_df['Mean CV Acc'],
              yerr=crit_df['Std'], color=['#388E3C', '#1976D2'],
              width=0.4, edgecolor='white', capsize=6)
for bar, val in zip(bars, crit_df['Mean CV Acc']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylim(0.6, 1.0)
ax.set_title('Gini vs Entropy Criterion', fontsize=12, fontweight='bold')
ax.set_ylabel('5-Fold CV Accuracy')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'dt_criterion_comparison.png'), dpi=120)
plt.show()

## 15. Algorithm Comparison

In [ ]:
comparison_data = {
    'Algorithm':  ['Logistic Regression', 'Random Forest', 'SVM', 'Neural Network', 'KNN', 'Decision Tree'],
    'Accuracy':   [0.8689, 0.8852, 0.8689, 0.8689, 0.8361, acc],
    'Precision':  [0.8125, 0.8800, 0.8400, 0.8400, 0.8125, prec],
    'Recall':     [0.9286, 0.8571, 0.8929, 0.9286, 0.9286, rec],
    'F1-Score':   [0.8667, 0.8684, 0.8657, 0.8667, 0.8571, f1],
    'ROC AUC':    [0.9513, 0.9600, 0.9400, 0.9450, 0.8800, auc],
}
comp_df = pd.DataFrame(comparison_data)
print('Algorithm Comparison:')
print(comp_df.to_string(index=False))

In [ ]:
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC AUC']
algorithms      = comp_df['Algorithm'].tolist()
palette         = ['#4CAF50', '#2196F3', '#FF9800', '#E91E63', '#9C27B0', '#795548']
x, width = np.arange(len(metrics_to_plot)), 0.13

fig, ax = plt.subplots(figsize=(15, 5))
for i, (algo, color) in enumerate(zip(algorithms, palette)):
    vals = [comp_df.loc[comp_df['Algorithm']==algo, m].values[0] for m in metrics_to_plot]
    ax.bar(x + i*width, vals, width, label=algo, color=color, alpha=0.85, edgecolor='white')

ax.set_xticks(x + width * 2.5)
ax.set_xticklabels(metrics_to_plot, fontsize=10)
ax.set_ylim(0.6, 1.05)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Algorithm Performance Comparison — Heart Disease Prediction',
             fontsize=12, fontweight='bold', pad=14)
ax.legend(fontsize=8, loc='lower right')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'dt_algorithm_comparison.png'), dpi=120)
plt.show()

## 16. Save Results

In [ ]:
dt_metrics = {
    'accuracy': acc, 'precision': prec, 'recall': rec,
    'f1_score': f1, 'roc_auc': auc, 'confusion_matrix': cm,
    'classification_report': classification_report(
        y_test, y_pred, target_names=['No Disease', 'Disease']
    )
}
metrics_path = os.path.join(RESULTS_DIR, 'dt_metrics.txt')
save_metrics(dt_metrics, metrics_path)
print(f'Metrics saved to: {metrics_path}')

## 17. Interpretation of Results

**Model Performance:**
- The Decision Tree achieves interpretable results with explicit decision rules
- The depth tuning plot shows the optimal balance between underfitting (shallow) and overfitting (deep)
- Feature importance reveals which clinical attributes are most informative

**Feature Importance Insights:**
- Top features (e.g., `ca`, `thal`, `cp`) align with known clinical risk factors for heart disease
- Features with zero importance were never used for any split — potential candidates for removal

**Clinical Interpretability:**
- The exported tree rules can be directly communicated to clinicians
- Each leaf node shows the class distribution and Gini impurity, aiding confidence estimation

## 18. Critical Analysis

### Strengths
- Highly interpretable — rules can be explained to clinical staff
- No feature scaling required
- Built-in feature importance
- Fast inference — O(log n)
- Handles non-linear boundaries naturally

### Limitations
- Prone to overfitting without pruning or depth constraints
- High variance — small changes in data can produce very different trees
- Axis-aligned splits — cannot model diagonal decision boundaries efficiently
- Not competitive with ensemble methods (Random Forest, XGBoost)

## 19. Future Improvements
- **Pruning (cost-complexity)**: Use `ccp_alpha` to post-prune the tree and improve generalisation
- **Random Forest**: Ensemble of Decision Trees drastically reduces variance
- **Gradient Boosting**: XGBoost/LightGBM for best-in-class tabular performance
- **SMOTE**: Address mild class imbalance
- **GridSearchCV**: Jointly optimise `max_depth`, `min_samples_split`, `min_samples_leaf`, `criterion`
- **Combine datasets**: Train on Cleveland + Hungarian + Switzerland + VA for richer generalisation

## 20. Individual Contribution (Member 6)

- Implemented Decision Tree with `sklearn.tree.DecisionTreeClassifier` (CART / Gini)
- No feature scaling applied — Decision Trees are scale-invariant
- Systematically tuned `max_depth` (1..15) using 5-fold CV
- Compared Gini vs Entropy splitting criteria
- Generated: Accuracy, Precision, Recall, F1, ROC-AUC, Sensitivity, Specificity
- Visualised the full decision tree and exported text decision rules
- Plotted Gini-based feature importance ranking
- Conducted 5-fold stratified cross-validation
- Compared Decision Tree against all other project algorithms
- Saved all results and plots to `results/` directory

**Files created:**
- `notebooks/06_decision_tree_analysis.ipynb` — This complete analysis notebook
- `results/dt_metrics.txt` — Saved evaluation metrics
- `results/dt_evaluation.png` — Confusion matrix & ROC curve
- `results/dt_tree_plot.png` — Full decision tree visualisation
- `results/dt_feature_importance.png` — Feature importance bar chart
- `results/dt_depth_tuning.png` — Depth tuning (train vs CV)
- `results/dt_cross_validation.png` — CV accuracy chart
- `results/dt_criterion_comparison.png` — Gini vs Entropy comparison
- `results/dt_algorithm_comparison.png` — Cross-algorithm comparison
- `results/dt_boxplots.png` — Feature box plots by class